# Train a "products in a room" LoRA on Stable Diffusion XL (Google Colab)

**How this compares to the SD1.5 and Qwen-Image notebooks:**

| | SD1.5 | Stable Diffusion XL | Qwen-Image |
|---|---|---|---|
| Base model | ~0.9B UNet | stable-diffusion-xl-base-1.0 | ~20B transformer |
| Fits on a free Colab T4 (16GB)? | Yes | Yes, tight (LoRA + fp16 + gradient checkpointing fits in ~14-15GB at 768px) | No |
| Minimum realistic GPU | T4 (free) | T4 GPU (free tier works, but is tight - an L4/A10 is more comfortable) | A100 40GB (Pro+) |
| Training recipe | hand-written loop (this notebook too) | hand-written loop (this notebook too) | driven through HF's own maintained script |

**What this does, same as the other notebooks in this folder:** removes the white studio background from your
supplier photos, pastes each product into a randomly generated room, captions it from the folder name, and trains
a LoRA on top of the frozen base model. Sections 1-4 (finding, validating, captioning and compositing the images)
are **identical** to the SD1.5 notebook — same data, same logic, copy-pasted rather than re-implemented so there is
exactly one place that logic lives to get right.

**Training objective: same as SD1.5** (predict the noise added at a random timestep, DDPM-style)
— SDXL did not change this, only the architecture and conditioning around it. If you've read the SD1.5 notebook's
training loop, the shape here will look familiar; the differences are called out in the comments.

## How to run it
1. Colab: **Runtime -> Change runtime type -> T4 GPU (free tier works, but is tight - an L4/A10 is more comfortable).**
2. Put your image folder in Google Drive, same structure as before (`ATRIANI/`, `CELADON/`, `LAZZONI/` (PDFs),
   `LUXUS/`...).
3. Leave `DRIVE_IMAGES_DIR = "auto"` in the **CONFIG** cell to search your Drive automatically.
4. **Run all.** Section 5 checks your GPU has at least ~14GB before continuing, and prints your measured
   seconds/step from the first few dozen steps so you have a real time estimate for your `MAX_TRAIN_STEPS`, not
   just the table below (which is a rough, unmeasured community-reported estimate):

| Steps | Estimated time |
|---|---|
| 500 | 20-35 min |
| 1000 | 40-70 min |
| 1500 (default) | 60-105 min |

5. The trained LoRA is written to `DRIVE_OUTPUT_DIR`, with checkpoints saved every `SAVE_EVERY` steps so a Colab
   disconnect doesn't cost you the whole run.

**Why SDXL over SD1.5.** SDXL's UNet is ~2.9x larger (2.6B vs 0.9B) and uses two text encoders
(OpenCLIP-ViT/G plus the original CLIP-ViT/L) instead of one, which generally gives noticeably better detail,
lighting and composition at the cost of needing more VRAM and a slightly slower per-step time. It is still a
DDPM/epsilon-prediction UNet underneath (same training objective as SD1.5), so this notebook's training loop is the
same shape as the SD1.5 one — the real differences are: two text encoders to run per caption, SDXL's extra
"micro-conditioning" inputs (`add_time_ids`: original size, crop coordinates, target size — SDXL was trained to
condition on these, and omitting them measurably hurts quality, so this notebook does not skip them), and a larger
default training resolution (768-1024 vs SD1.5's 512) to match what SDXL was actually trained on.


In [ ]:
# ============================== CONFIG: edit these ==============================
DRIVE_IMAGES_DIR = "/content/drive/MyDrive/Curalina - Humber Capstone/Supplier Images"   # "auto" = search your Drive + shared drives for the supplier folders.
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/curalina_sdxl_lora_output"

LOCAL_IMAGES_DIR = "/Users/rjsalmon/Documents/Humber/misc.curalina/Supplier Images"
LOCAL_OUTPUT_DIR = "./sdxl_lora_output"

BASE_MODEL_ID = "stabilityai/stable-diffusion-xl-base-1.0"
TRIGGER = "crlnstyle"
RESOLUTION = 768
MAX_TRAIN_STEPS = 1500
BATCH_SIZE = 1
LEARNING_RATE = 1e-4
LORA_RANK = 16
SAVE_EVERY = 300
GRADIENT_CHECKPOINTING = True
QUANTIZE_4BIT = False   # SDXL's 2.6B UNet fits in fp16 without quantization on a 16GB+ GPU; 4-bit isn't needed here.
P_ROOM_BACKGROUND = 0.8
MIN_SIDE_PX = 200
SEED = 42

# Debug: set to a small number (e.g. 20) to try the pipeline on a few images first, or None for everything.
LIMIT_IMAGES = None


In [ ]:
import os, sys, json, random, math, shutil, re, time, difflib
from pathlib import Path
from collections import Counter

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")


def explain_missing(path: Path) -> str:
    # Finds the deepest folder that DOES exist, lists what is in it, and suggests the closest spelling.
    p = Path(path)
    missing = []
    while not p.exists() and p != p.parent:
        missing.insert(0, p.name)
        p = p.parent
    try:
        children = sorted(os.listdir(p))
    except OSError:
        children = []
    msg = f"'{path}' was not found.\nThe part that exists is: {p}\nThe first name that is missing is: '{missing[0] if missing else ''}'\n"
    msg += f"Folders that DO exist inside {p}:\n  " + ("\n  ".join(children[:40]) if children else "(none - is this the Google account that has access?)") + "\n"
    if missing and children:
        close = difflib.get_close_matches(missing[0], children, n=3, cutoff=0.4)
        if close:
            msg += f"Did you mean: {close}\n"
    msg += "Tip: in Colab open the Files panel (folder icon, left), browse to the folder, right-click -> Copy path, and paste it into the CONFIG cell."
    return msg


SUPPLIER_MARKERS = {"atriani", "celadon", "lazzoni", "luxus"}


def find_images_root(max_depth: int = 5):
    # Looks through every shared drive and My Drive for a folder that contains at least two of the supplier folders.
    roots = []
    shared = Path("/content/drive/Shareddrives")
    if shared.is_dir():
        roots += [shared / d for d in sorted(os.listdir(shared))]
    roots.append(Path("/content/drive/MyDrive"))
    hits = []
    for base in roots:
        base_parts = len(base.parts)
        for cur, dirs, _ in os.walk(base):
            depth = len(Path(cur).parts) - base_parts
            dirs[:] = [d for d in dirs if not d.startswith(".")] if depth < max_depth else []
            if len({d.lower() for d in dirs} & SUPPLIER_MARKERS) >= 2:
                hits.append(Path(cur))
                dirs[:] = []                       # found it; do not search inside
    return hits


if IN_COLAB:
    # Uncomment if a package is missing. Colab already has torch/torchvision.
    # !pip -q install diffusers transformers accelerate peft
    from google.colab import drive
    drive.mount("/content/drive",force_remount=True)
    for root in ["/content/drive", "/content/drive/Shareddrives", "/content/drive/MyDrive"]:
        print(f"{root} ->", sorted(os.listdir(root))[:30] if os.path.isdir(root) else "(not found)")

    # Make sure to run the CONFIG cell (cell b568965d) first to set DRIVE_IMAGES_DIR correctly!
    print(f"DEBUG (Cell 6f9d2616): DRIVE_IMAGES_DIR is '{DRIVE_IMAGES_DIR}'")

    if DRIVE_IMAGES_DIR.strip().lower() == "auto":
        print("Searching your Drive for the supplier image folders (looks for LUXUS / CELADON / ATRIANI / LAZZONI)...")
        hits = find_images_root()
        if len(hits) == 1:
            IMAGES_DIR = hits[0]
            print("Found:", IMAGES_DIR)
        elif len(hits) > 1:
            raise SystemExit("Found several candidate folders - copy one into DRIVE_IMAGES_DIR:\n  " + "\n  ".join(map(str, hits)))
        else:
            raise SystemExit("Could not find a folder containing LUXUS / CELADON / ATRIANI / LAZZONI anywhere in your Drive.\n"
                             "Check you mounted the Google account that has access, and that the shared drive is listed above.")
    else:
        IMAGES_DIR = Path(DRIVE_IMAGES_DIR)
    OUTPUT_DIR = Path(DRIVE_OUTPUT_DIR)
    WORK_DIR = Path("/content/work")

else:
    IMAGES_DIR = Path(LOCAL_IMAGES_DIR)
    OUTPUT_DIR = Path(LOCAL_OUTPUT_DIR)
    WORK_DIR = Path("./work")

# Check the input folder FIRST, with a readable message, before creating anything.
print(f"DEBUG (Cell 6f9d2616): IMAGES_DIR is '{IMAGES_DIR}' before checking existence.")
if not IMAGES_DIR.is_dir():
    raise SystemExit("IMAGE FOLDER PROBLEM\n" + explain_missing(IMAGES_DIR))

*(Identical to the SD1.5 notebook from here through Section 4 - same data, same logic.)*

## 1. Find the images (and skip everything else)

We walk the whole folder tree. Only these extensions are treated as images; anything else — the Lazzoni PDFs,
`.DS_Store`, spreadsheets — is counted and skipped. Then every candidate is opened once with Pillow: files that are
corrupt, unreadable or smaller than `MIN_SIDE_PX` are dropped and listed, so training never hits a bad file.


In [ ]:
import pandas as pd
from PIL import Image, UnidentifiedImageError

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".webp"}
Image.MAX_IMAGE_PIXELS = 60_000_000  # guard against decompression bombs

found, skipped_non_image = [], Counter()
for root, dirs, files in os.walk(IMAGES_DIR):
    dirs[:] = [d for d in dirs if not d.startswith(".")]
    for name in files:
        if name.startswith("."):
            continue
        p = Path(root) / name
        ext = p.suffix.lower()
        if ext in IMAGE_EXTS:
            found.append(p)
        else:
            skipped_non_image[ext or "(no extension)"] += 1

print(f"{len(found)} candidate image files")
print("skipped non-image files by type:", dict(skipped_non_image) or "none")

valid, rejected = [], []
for p in sorted(found):
    try:
        with Image.open(p) as im:
            im.verify()                      # cheap integrity check
        with Image.open(p) as im:            # verify() invalidates the handle, reopen for size
            w, h = im.size
        if min(w, h) < MIN_SIDE_PX:
            rejected.append((str(p.relative_to(IMAGES_DIR)), f"too small ({w}x{h})"))
        else:
            valid.append(p)
    except (UnidentifiedImageError, OSError, ValueError, Image.DecompressionBombError) as e:
        rejected.append((str(p.relative_to(IMAGES_DIR)), f"unreadable: {type(e).__name__}"))

if LIMIT_IMAGES:
    random.Random(SEED).shuffle(valid)
    valid = sorted(valid[:LIMIT_IMAGES])

print(f"{len(valid)} usable images, {len(rejected)} rejected")
pd.DataFrame(rejected, columns=["file", "reason"]).head(15)


## 2. Captions from the folder structure

Supplier = the top-level folder (`LUXUS`, `CELADON`, ...). Product name = the folder the image sits in, with the
`001 - ` numbering and any trailing SKU removed. A keyword pass maps names to a furniture category and a sensible room
(a bed goes in a bedroom, a dining chair in a dining room). Nothing here needs the spreadsheets.


In [ ]:
CATEGORY_RULES = [  # first match wins; (category, regex, rooms it belongs in)
    ("dining chair", r"dining chair", ["dining room"]),
    ("dining table", r"dining table", ["dining room"]),
    ("accent chair", r"accent chair|armchair|lounge chair|poltrona|chair", ["living room", "bedroom"]),
    ("coffee table", r"coffee table|cocktail table", ["living room"]),
    ("side table", r"side table|end table", ["living room", "bedroom"]),
    ("bar cabinet", r"bar cabinet", ["dining room", "living room"]),
    ("sectional sofa", r"sectional|modular", ["living room"]),
    ("sofa", r"\bsofa\b|\bloveseat\b", ["living room"]),
    ("ottoman", r"ottoman", ["living room"]),
    ("bench", r"\bbench\b", ["bedroom", "living room"]),
    ("nightstand", r"night ?stand|bedside", ["bedroom"]),
    ("bed", r"\bbed\b|\bking\b|\bqueen\b", ["bedroom"]),
    ("sideboard", r"sideboard|buffet|credenza", ["dining room", "living room"]),
    ("table", r"table", ["dining room", "living room"]),
    ("console or cabinet", r"console|media|cabinet|shelf|bookcase|vanity|desk|stool", ["living room", "dining room"]),
]


def clean_name(folder_name: str) -> str:
    n = re.sub(r"^\s*\d+\s*-\s*", "", folder_name)          # "001 - Halo Nightstand" -> "Halo Nightstand"
    n = re.sub(r"\s*-\s*\d{4,}\s*$", "", n)                # Celadon "Traverse I - 21368" -> "Traverse I"
    n = n.replace("_", ":").strip()
    return n or folder_name


def describe(path: Path):
    rel = path.relative_to(IMAGES_DIR)
    parts = rel.parts
    supplier = parts[0] if len(parts) > 1 else "unknown"
    folder = parts[-2] if len(parts) > 2 else path.stem        # product folder; fall back to file name
    if folder.lower() in {"product images", supplier.lower()}:
        folder = path.stem
    name = clean_name(folder)
    if supplier.upper() == "CELADON":
        return dict(supplier=supplier, name=name, category="framed wall art", rooms=["living room", "bedroom"], is_art=True)
    for category, pattern, rooms in CATEGORY_RULES:
        if re.search(pattern, name.lower()):
            return dict(supplier=supplier, name=name, category=category, rooms=rooms, is_art=False)
    return dict(supplier=supplier, name=name, category="piece of furniture", rooms=["living room"], is_art=False)


records = []
for p in valid:
    d = describe(p)
    d["path"] = str(p)
    records.append(d)

df = pd.DataFrame(records)
print(df.groupby(["supplier", "category"]).size().to_string())
df[["supplier", "name", "category"]].sample(min(8, len(df)), random_state=SEED)


## 3. Cut out the products and build "product in a room" training images

Studio photos have a white background, which would teach the model "furniture floats on white". So we **remove the
background** (flood-fill from the image border — white areas *inside* the product, like a white sofa, are kept) and paste
the cutout into a randomly generated room: random wall colour, floor, baseboard, sometimes a window, with a soft shadow.
Artwork is hung on the wall instead of placed on the floor. Cutouts are cached in `WORK_DIR`.


In [ ]:
import numpy as np
from PIL import ImageDraw, ImageFilter

CUTOUT_DIR = WORK_DIR / "cutouts"
CUTOUT_DIR.mkdir(parents=True, exist_ok=True)


def matte_white_background(img: Image.Image, tolerance: int = 45, max_side: int = 640) -> Image.Image:
    rgba = img.convert("RGBA")
    if max(rgba.size) > max_side:
        rgba.thumbnail((max_side, max_side), Image.Resampling.LANCZOS)
    flat = Image.alpha_composite(Image.new("RGBA", rgba.size, (255, 255, 255, 255)), rgba).convert("RGB")
    work = flat.copy()
    w, h = work.size
    marker = (255, 0, 255)
    floor = 255 - tolerance
    for seed in [(0, 0), (w - 1, 0), (0, h - 1), (w - 1, h - 1), (w // 2, 0), (w // 2, h - 1), (0, h // 2), (w - 1, h // 2)]:
        px = flat.getpixel(seed)
        if all(int(c) >= floor for c in px[:3]):
            ImageDraw.floodfill(work, seed, marker, thresh=tolerance)
    arr = np.array(rgba)
    background = np.all(np.array(work) == np.array(marker), axis=2)
    arr[..., 3][background] = 0
    arr[..., 3][arr[..., 3] < 40] = 0
    alpha = Image.fromarray(arr[..., 3]).filter(ImageFilter.MinFilter(3))   # erode 1px: drops the white fringe
    arr[..., 3] = np.array(alpha)
    return Image.fromarray(arr, "RGBA")


SKIP_REASONS = Counter()
SWATCH_NAME = re.compile(r"^(l\d+|ai\d+)\b", re.I)   # finish/material swatch files like "L139.png", "AI01- Chrome.webp"


def build_cutout(path: str, is_art: bool):
    # Returns the cached cutout path, or None (and counts why) if the image is not a usable product photo.
    p = Path(path)
    if SWATCH_NAME.match(p.stem):
        SKIP_REASONS["finish swatch (by file name)"] += 1
        return None
    out = CUTOUT_DIR / (re.sub(r"[^A-Za-z0-9]+", "_", str(p.relative_to(IMAGES_DIR))) + ".png")
    if out.exists():
        return str(out)
    try:
        with Image.open(p) as im:
            cut = matte_white_background(im)
        alpha = np.array(cut)[..., 3]
        visible = (alpha > 127).mean()
        if visible < 0.02:
            SKIP_REASONS["almost all background"] += 1
            return None
        if not is_art and visible > 0.93:
            SKIP_REASONS["no studio background (swatch / close-up)"] += 1
            return None
        rgb = np.array(cut.convert("RGB")).astype(float)
        if not is_art and rgb[alpha > 127].mean() > 240:      # mostly white pixels: a line drawing, not a photo
            SKIP_REASONS["line drawing (white on white)"] += 1
            return None
        cut.crop(cut.getchannel("A").getbbox()).save(out)
        return str(out)
    except Exception as e:                        # never let one bad file stop the run
        SKIP_REASONS[f"error: {type(e).__name__}"] += 1
        return None


t0 = time.time()
df["cutout"] = [build_cutout(p, a) for p, a in zip(df["path"], df["is_art"])]
before = len(df)
df = df[df["cutout"].notna()].reset_index(drop=True)      # only clean product cutouts are trained on
print(f"{len(df)}/{before} images kept after cutout ({time.time() - t0:.0f}s). Dropped: {dict(SKIP_REASONS) or 'none'}")

WALLS = [(236, 231, 222), (222, 214, 200), (205, 210, 205), (196, 187, 175), (233, 224, 214),
         (94, 98, 96), (58, 56, 60), (170, 150, 128), (215, 220, 224)]
FLOORS = [(176, 140, 100), (140, 105, 75), (205, 190, 165), (95, 80, 70), (160, 150, 140), (60, 55, 52)]


def make_room_background(rng: random.Random, size: int = 512):
    wall, floor = rng.choice(WALLS), rng.choice(FLOORS)
    horizon = int(size * rng.uniform(0.58, 0.68))
    img = Image.new("RGB", (size, size), wall)
    d = ImageDraw.Draw(img)
    d.rectangle([0, horizon, size, size], fill=floor)
    d.rectangle([0, horizon - 10, size, horizon], fill=tuple(min(255, c + 25) for c in wall))   # baseboard
    if rng.random() < 0.5:                                                                      # window
        x0 = rng.randint(int(size * 0.08), int(size * 0.55)); w = rng.randint(90, 150)
        d.rectangle([x0, int(size * 0.12), x0 + w, horizon - 60], fill=(250, 250, 244), outline=(120, 120, 110), width=4)
    grad = np.linspace(1.06, 0.92, size)[:, None, None]
    return Image.fromarray(np.clip(np.array(img) * grad, 0, 255).astype("uint8")), horizon


def compose_room_sample(cut: Image.Image, is_art: bool, rng: random.Random, size: int = 512):
    bg, horizon = make_room_background(rng, size)
    cw, ch = cut.size
    if is_art:
        scale = rng.uniform(0.28, 0.5) * size / max(cw, ch)
    else:
        scale = rng.uniform(0.45, 0.85) * size / max(cw, ch)
    cut = cut.resize((max(1, int(cw * scale)), max(1, int(ch * scale))), Image.Resampling.LANCZOS)
    cw, ch = cut.size
    x = int(size / 2 - cw / 2 + rng.uniform(-0.12, 0.12) * size)
    if is_art:
        y = int(horizon * rng.uniform(0.12, 0.35))
    else:
        y = min(size - ch, int(horizon + (size - horizon) * rng.uniform(0.15, 0.55)) - ch)
        shadow = Image.new("RGBA", bg.size, (0, 0, 0, 0))
        ImageDraw.Draw(shadow).ellipse([x, y + ch - int(ch * 0.08), x + cw, y + ch + int(ch * 0.1)], fill=(0, 0, 0, 90))
        bg = Image.alpha_composite(bg.convert("RGBA"), shadow.filter(ImageFilter.GaussianBlur(9))).convert("RGB")
    bg.paste(cut, (x, max(0, y)), cut)
    return bg


def plain_product_sample(cutout_path: str, size: int = 512):
    with Image.open(cutout_path) as im:
        im = im.convert("RGBA")
        flat = Image.alpha_composite(Image.new("RGBA", im.size, (255, 255, 255, 255)), im).convert("RGB")
    flat.thumbnail((size, size), Image.Resampling.LANCZOS)
    canvas = Image.new("RGB", (size, size), (255, 255, 255))
    canvas.paste(flat, ((size - flat.width) // 2, (size - flat.height) // 2))
    return canvas


def make_sample(row, rng: random.Random, size: int = 512):
    """Returns (PIL image, caption): a product pasted into a synthetic room, or a plain white-background product photo."""
    label = f"{row['name']}" if row["category"] in row["name"].lower() else f"{row['name']} {row['category']}"
    if rng.random() < P_ROOM_BACKGROUND:
        room = rng.choice(row["rooms"])
        with Image.open(row["cutout"]) as c:
            image = compose_room_sample(c.convert("RGBA"), row["is_art"], rng, size)
        caption = f"{TRIGGER}, a {label} in a {room}, interior design photo"
    else:
        image = plain_product_sample(row["cutout"], size)
        caption = f"{TRIGGER}, product photo of a {label}, white background"
    return image, caption


In [ ]:
import matplotlib.pyplot as plt

rng_preview = random.Random(SEED)
sample_rows = df.sample(min(8, len(df)), random_state=SEED)
fig, axes = plt.subplots(2, 4, figsize=(16, 9))
for ax, (_, row) in zip(axes.ravel(), sample_rows.iterrows()):
    img, cap = make_sample(row, rng_preview)
    ax.imshow(img); ax.axis("off"); ax.set_title(cap[:48] + "...", fontsize=8)
fig.suptitle("Training samples (background-removed products pasted into synthetic rooms) - same data as the SD1.5 run")
fig.tight_layout(); plt.show()


## 5. GPU check, then load SDXL and attach a LoRA

SDXL needs two text encoders (`text_encoder` + `text_encoder_2`, with their matching tokenizers) instead of SD1.5's
one, and the UNet is conditioned on `add_time_ids` (original image size, the crop offset used when preparing the
training image, and the target size) in addition to the text embedding — SDXL was trained expecting these, and
quality suffers if you just leave them out. Since every training image here is a clean square crop with no
cropping-augmentation, `crop_coords` is simply `(0, 0)` throughout.


In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPTextModel, CLIPTextModelWithProjection, CLIPTokenizer
from diffusers import AutoencoderKL, DDPMScheduler, UNet2DConditionModel, StableDiffusionXLPipeline
from diffusers.optimization import get_scheduler
from peft import LoraConfig
from peft.utils import get_peft_model_state_dict
from diffusers.utils import convert_state_dict_to_diffusers

if not torch.cuda.is_available():
    raise SystemExit("No GPU found. In Colab: Runtime > Change runtime type > T4 GPU (or better), then run all again.")
device = torch.device("cuda")
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"GPU: {torch.cuda.get_device_name(0)}  ({total_vram_gb:.0f} GB)")
if total_vram_gb < 14:
    raise SystemExit(
        f"This GPU reports {total_vram_gb:.0f} GB. SDXL LoRA training at {RESOLUTION}px needs roughly 14GB+ even "
        f"with gradient checkpointing. Try Runtime > Change runtime type for a bigger GPU, or lower RESOLUTION to "
        f"512 in the CONFIG cell (SDXL still works at 512, just was trained mainly at 768-1024)."
    )
torch.manual_seed(SEED); random.seed(SEED); np.random.seed(SEED)

tokenizer = CLIPTokenizer.from_pretrained(BASE_MODEL_ID, subfolder="tokenizer")
tokenizer_2 = CLIPTokenizer.from_pretrained(BASE_MODEL_ID, subfolder="tokenizer_2")
text_encoder = CLIPTextModel.from_pretrained(BASE_MODEL_ID, subfolder="text_encoder", torch_dtype=torch.float16).to(device)
text_encoder_2 = CLIPTextModelWithProjection.from_pretrained(BASE_MODEL_ID, subfolder="text_encoder_2", torch_dtype=torch.float16).to(device)
vae = AutoencoderKL.from_pretrained(BASE_MODEL_ID, subfolder="vae", torch_dtype=torch.float32).to(device)  # SDXL's vae is numerically sensitive in fp16
unet = UNet2DConditionModel.from_pretrained(BASE_MODEL_ID, subfolder="unet").to(device)
noise_scheduler = DDPMScheduler.from_pretrained(BASE_MODEL_ID, subfolder="scheduler")

text_encoder.requires_grad_(False); text_encoder_2.requires_grad_(False); vae.requires_grad_(False); unet.requires_grad_(False)
unet.add_adapter(LoraConfig(r=LORA_RANK, lora_alpha=LORA_RANK, init_lora_weights="gaussian",
                            target_modules=["to_k", "to_q", "to_v", "to_out.0"]))
if GRADIENT_CHECKPOINTING:
    unet.enable_gradient_checkpointing()

lora_params = [p for p in unet.parameters() if p.requires_grad]
print(f"trainable LoRA parameters: {sum(p.numel() for p in lora_params):,} (of {sum(p.numel() for p in unet.parameters()):,} total)")


def encode_prompt(captions):
    ids1 = tokenizer(captions, padding="max_length", max_length=tokenizer.model_max_length, truncation=True, return_tensors="pt").input_ids.to(device)
    ids2 = tokenizer_2(captions, padding="max_length", max_length=tokenizer_2.model_max_length, truncation=True, return_tensors="pt").input_ids.to(device)
    with torch.no_grad():
        out1 = text_encoder(ids1, output_hidden_states=True)
        out2 = text_encoder_2(ids2, output_hidden_states=True)
    prompt_embeds = torch.cat([out1.hidden_states[-2], out2.hidden_states[-2]], dim=-1)   # SDXL concatenates both encoders' penultimate layer
    pooled_embeds = out2[0]                                                                # the projection ("pooled") output, text_encoder_2 only
    return prompt_embeds, pooled_embeds


class ProductRoomDataset(Dataset):
    def __init__(self, frame, size=RESOLUTION):
        self.rows = frame.to_dict("records"); self.size = size
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        rng = random.Random()
        try:
            image, caption = make_sample(self.rows[i], rng, self.size)
        except Exception:
            return None
        pixels = torch.from_numpy(np.array(image, dtype=np.float32) / 127.5 - 1.0).permute(2, 0, 1)
        return {"pixel_values": pixels, "caption": caption}


def collate(batch):
    batch = [b for b in batch if b is not None]
    if not batch:
        return None
    return {"pixel_values": torch.stack([b["pixel_values"] for b in batch]), "captions": [b["caption"] for b in batch]}


train_dl = DataLoader(ProductRoomDataset(df), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate,
                      num_workers=2, drop_last=len(df) >= BATCH_SIZE)
optimizer = torch.optim.AdamW(lora_params, lr=LEARNING_RATE, weight_decay=1e-2)
lr_scheduler = get_scheduler("cosine", optimizer=optimizer, num_warmup_steps=50, num_training_steps=MAX_TRAIN_STEPS)
scaler = torch.amp.GradScaler("cuda")
print(f"{len(df)} images, {len(train_dl)} batches/epoch, {MAX_TRAIN_STEPS} steps (~{MAX_TRAIN_STEPS / max(1, len(train_dl)):.1f} epochs)")


## 6. Training loop

Same `zero_grad -> forward -> loss -> backward -> step` shape as SD1.5, with two differences: captions go through
both text encoders (`encode_prompt` above), and the UNet call passes `added_cond_kwargs` with the pooled embedding
and `add_time_ids` SDXL expects. A running measured steps/sec prints every 25 steps so you can sanity-check the
time-table estimate in Section 0 against what you're actually getting.


In [ ]:
def save_lora(path: Path):
    path.mkdir(parents=True, exist_ok=True)
    state = convert_state_dict_to_diffusers(get_peft_model_state_dict(unet))
    StableDiffusionXLPipeline.save_lora_weights(save_directory=str(path), unet_lora_layers=state, safe_serialization=True)


def make_add_time_ids(bsz):
    # (original_size, crop_top_left, target_size) per sample - every training image here is a clean RESOLUTION
    # square with no crop augmentation, so crop_top_left is always (0, 0).
    add_time_ids = torch.tensor([[RESOLUTION, RESOLUTION, 0, 0, RESOLUTION, RESOLUTION]] * bsz, device=device, dtype=torch.float16)
    return add_time_ids


losses, lrs = [], []
step = 0
unet.train()
t0 = time.time()
try:
    while step < MAX_TRAIN_STEPS:
        for batch in train_dl:
            if batch is None:
                continue
            pixel_values = batch["pixel_values"].to(device, dtype=torch.float32)  # fp32 vae per above
            prompt_embeds, pooled_embeds = encode_prompt(batch["captions"])

            with torch.no_grad():
                latents = (vae.encode(pixel_values).latent_dist.sample() * vae.config.scaling_factor).to(torch.float16)
            noise = torch.randn_like(latents)
            timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (latents.shape[0],), device=device).long()
            noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

            optimizer.zero_grad()
            with torch.autocast("cuda", dtype=torch.float16):
                noise_pred = unet(
                    noisy_latents, timesteps, prompt_embeds.to(torch.float16),
                    added_cond_kwargs={"text_embeds": pooled_embeds.to(torch.float16), "time_ids": make_add_time_ids(latents.shape[0])},
                ).sample
            loss = F.mse_loss(noise_pred.float(), noise.float())
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(lora_params, 1.0)
            scaler.step(optimizer)
            scaler.update()
            lr_scheduler.step()

            step += 1
            losses.append(loss.item()); lrs.append(lr_scheduler.get_last_lr()[0])
            if step % 25 == 0:
                elapsed = time.time() - t0
                print(f"Step: {step}/{MAX_TRAIN_STEPS}, Loss: {np.mean(losses[-25:]):.4f}, LR: {lrs[-1]:.2e}, "
                      f"elapsed: {elapsed/60:.1f} min, {elapsed/step:.2f} sec/step, "
                      f"projected total: {elapsed/step*MAX_TRAIN_STEPS/60:.0f} min")
            if step % SAVE_EVERY == 0:
                save_lora(OUTPUT_DIR / f"checkpoint-{step}")
            if step >= MAX_TRAIN_STEPS:
                break
except KeyboardInterrupt:
    print("Interrupted - saving what has been trained so far.")

save_lora(OUTPUT_DIR / "final")
pd.DataFrame({"loss": losses, "lr": lrs}).to_csv(OUTPUT_DIR / "training_log.csv", index=False)
with open(OUTPUT_DIR / "training_config.json", "w") as f:
    json.dump(dict(base_model=BASE_MODEL_ID, trigger=TRIGGER, steps=step, rank=LORA_RANK, resolution=RESOLUTION,
                   batch_size=BATCH_SIZE, lr=LEARNING_RATE, images=int(len(df)),
                   suppliers=sorted(df["supplier"].unique().tolist())), f, indent=2)
print("saved LoRA to", OUTPUT_DIR / "final")


## 7. Loss and learning-rate plots

In [ ]:
def moving_average(x, k=25):
    return np.convolve(x, np.ones(k) / k, mode="valid")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(losses, alpha=0.3); axes[0].plot(range(24, len(losses)), moving_average(losses)); axes[0].set_title("Loss"); axes[0].set_xlabel("step")
axes[1].plot(lrs); axes[1].set_title("Learning rate"); axes[1].set_xlabel("step")
fig.tight_layout(); plt.show()


## 8. Save for download

In [ ]:
zip_path = shutil.make_archive(str(OUTPUT_DIR / "curalina_sdxl_lora"), "zip", str(OUTPUT_DIR / "final"))
print("zipped to", zip_path)


## 9. Use it: plain model vs. model + your LoRA

Frees the training components first, since holding the full training state (optimizer, both text encoders'
gradients-disabled copies, etc.) and a fresh inference pipeline in VRAM at once is unnecessary memory pressure.


In [ ]:
del unet, text_encoder, text_encoder_2, vae, optimizer
torch.cuda.empty_cache()

pipe = StableDiffusionXLPipeline.from_pretrained(BASE_MODEL_ID, torch_dtype=torch.float16).to(device)
compare_prompt = f"{TRIGGER}, a sofa and a coffee table in a bright living room, interior design photo"

gen = torch.Generator(device=device).manual_seed(SEED)
base_image = pipe(compare_prompt, num_inference_steps=30, generator=gen).images[0]

pipe.load_lora_weights(str(OUTPUT_DIR / "final"))
gen = torch.Generator(device=device).manual_seed(SEED)
lora_image = pipe(compare_prompt, num_inference_steps=30, generator=gen).images[0]

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
axes[0].imshow(base_image); axes[0].axis("off"); axes[0].set_title("Base SDXL")
axes[1].imshow(lora_image); axes[1].axis("off"); axes[1].set_title("Base + your LoRA")
fig.suptitle(compare_prompt)
fig.tight_layout(); plt.show()


## 10. Try your own prompt


In [ ]:
my_prompt = f"{TRIGGER}, a boucle sofa and a walnut coffee table in a bright living room, interior design photo"
gen = torch.Generator(device=device).manual_seed(123)
display(pipe(my_prompt, num_inference_steps=30, generator=gen).images[0])


## Using the result later

Same pattern as the SD1.5 LoRA: the room-rendering service would need an `SDXL_LORA_PATH`-style env var analogous
to `sd15`'s `SD15_LORA_PATH` (see `ai_services/renderers/sd15/src/curalina_sd15/pipeline.py` for the pattern a new
`sdxl` renderer sidecar would follow — none exists yet in this repo).
